# Disaster Zone Aerial Assessment System
### Powered by TwelveLabs Marengo + Pegasus

This notebook analyzes drone footage of disaster-affected areas to produce:
1. **Damage Assessment** — identifies severely impacted structures and infrastructure
2. **Resource Manifest** — maps damage types to required emergency supplies
3. **Access & Route Analysis** — identifies obstructions and viable alternate routes
4. **Financial Damage Estimate** — cost estimate based on observed damage

**Models used:**
- `marengo3.0` — semantic video search for scene-level damage location
- `pegasus1.2` — video-language model for detailed natural language analysis

In [1]:
# # Install TwelveLabs SDK if not already installed
# %pip install twelvelabs python-dotenv rich --quiet

In [2]:
import os
import json
import re
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

from twelvelabs import TwelveLabs
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markdown import Markdown
from rich import print as rprint

load_dotenv(find_dotenv(usecwd=True))
console = Console()

## Configuration
Set your TwelveLabs API key and video source below.
- Set `VIDEO_PATH` to a local file path **or** set `VIDEO_URL` to a publicly accessible video URL.
- Leave `VIDEO_PATH = None` to use the URL instead.

In [3]:
# --- CONFIGURATION ---
API_KEY = os.getenv("TWELVELABS_API_KEY", "YOUR_API_KEY_HERE")

# Provide either a local file path OR a public URL to the drone footage
VIDEO_PATH = r"downloads\STL_tornado.mp4"          # e.g. r"C:\Videos\tornado_survey.mp4"
VIDEO_URL  = None          # e.g. "https://example.com/drone_footage.mp4"

# Optional: disaster type hint improves prompt context
# One of: "tornado", "hurricane", "wildfire", "flood", "earthquake", "auto"
DISASTER_TYPE = "tornado"

# City/region for location-aware analysis — used to ground street name extraction
# e.g. "St. Louis, MO", "Miami, FL", "Joplin, MO"
LOCATION_CITY = "St. Louis, MO"

# Label used in the report header (can be more specific than LOCATION_CITY)
LOCATION_LABEL = LOCATION_CITY

# Index name — reuse an existing index name to skip re-indexing the same video
INDEX_NAME = "disaster_assessment_index"
# ----------------------

assert API_KEY != "YOUR_API_KEY_HERE", "Set your TwelveLabs API key above or in a .env file"
assert VIDEO_PATH or VIDEO_URL, "Provide VIDEO_PATH or VIDEO_URL"

client = TwelveLabs(api_key=API_KEY)
console.print("TwelveLabs client initialized")

# --- Default values for all analysis outputs ---
# Each is overwritten by its respective cell when it runs.
LOCATION_CONTEXT = LOCATION_CITY
geo_items        = []
damage_items     = []
resource_items   = []
access_data      = {"access_points": [], "staging_areas": []}
marengo_results  = {}
damage_counts    = {}
cost_rows        = []
total_low        = 0.0
total_mid        = 0.0
total_high       = 0.0

TwelveLabs client initialized

## Step 1 — Create or Retrieve Index
We create one index with **both** models enabled:
- Marengo handles semantic scene search
- Pegasus handles video-to-text generation

In [4]:
def get_or_create_index(client: TwelveLabs, name: str):
    """Return existing index by name or create a new one."""
    for idx in client.indexes.list():
        if idx.index_name == name:
            console.print(f"[yellow]Reusing existing index:[/yellow] {idx.id}")
            return idx

    console.print(f"[blue]Creating new index:[/blue] {name}")
    idx = client.indexes.create(
        index_name=name,
        models=[
            {
                "model_name": "marengo3.0",
                "model_options": ["visual", "audio"]
            },
            {
                "model_name": "pegasus1.2",
                "model_options": ["visual", "conversation"]
            }
        ]
    )
    console.print(f"[green]Index created:[/green] {idx.id}")
    return idx


index = get_or_create_index(client, INDEX_NAME)
console.print(f"Index ID: [bold]{index.id}[/bold]")

Reusing existing index: 69ed14a5dc6eea3470b348af

Index ID: 69ed14a5dc6eea3470b348af

## Step 2 — Upload & Index the Drone Footage

In [5]:
def upload_video(client: TwelveLabs, index_id: str, path: str | None, url: str | None):
    """Upload video from local path or URL and return the completed task."""
    console.print("[blue]Uploading video for indexing...[/blue]")

    if path:
        with open(path, "rb") as f:
            task = client.tasks.create(index_id=index_id, video_file=f)
    else:
        task = client.tasks.create(index_id=index_id, video_url=url)

    console.print(f"Task created: [bold]{task.id}[/bold]  |  Waiting for indexing to complete...")

    def on_progress(t):
        console.print(f"  Status: {t.status}")

    completed = client.tasks.wait_for_done(task.id, sleep_interval=10, callback=on_progress)

    if completed.status != "ready":
        raise RuntimeError(f"Indexing failed with status: {completed.status}")

    console.print(f"[green]Video indexed successfully![/green]  Video ID: [bold]{completed.video_id}[/bold]")
    return completed


task = upload_video(client, index.id, VIDEO_PATH, VIDEO_URL)
VIDEO_ID = task.video_id

Uploading video for indexing...

Task created: 69ed3a86c0f9af9d63992ae2  |  Waiting for indexing to complete...

Status: ready

Video indexed successfully!  Video ID: 69ed3a86c0f9af9d63992ae2

## Step 3 — Location Identification (Pegasus)
Extracts visible street signs, intersections, landmarks, and address numbers from the footage to anchor all subsequent analysis to real-world locations.

In [6]:
GEO_PROMPT = f"""
You are analyzing aerial drone footage captured over {LOCATION_CITY} after a disaster.

Scan the entire video for any location identifiers visible in the footage — street signs,
intersection signs, neighborhood markers, recognizable landmarks, or address numbers.

For each one found, provide:
- TYPE: street_sign | intersection | neighborhood | landmark | address_number
- IDENTIFIER: The exact text or name visible
- CONFIDENCE: high | medium | low
- TIMESTAMP_APPROX: When it appears (e.g., "0:05-0:12")
- NOTES: Any context helpful for placing it on a map of {LOCATION_CITY}

Format as a JSON array of objects with keys: type, identifier, confidence, timestamp_approx, notes
Only output the JSON array. No preamble or commentary.
"""

console.print("[blue]Running location identification with Pegasus...[/blue]")
geo_response = client.analyze(video_id=VIDEO_ID, prompt=GEO_PROMPT)
geo_text = geo_response.data

json_match = re.search(r'\[.*\]', geo_text, re.DOTALL)
if json_match:
    geo_items = json.loads(json_match.group())
else:
    geo_items = []

high_confidence = [
    g["identifier"] for g in geo_items
    if isinstance(g, dict) and g.get("confidence") in ("high", "medium")
]
LOCATION_CONTEXT = (
    f"{LOCATION_CITY}. Known visible locations: {', '.join(high_confidence)}"
    if high_confidence else LOCATION_CITY
)

console.print(f"[green]{len(geo_items)} location marker(s) identified[/green]")
console.print(f"Location context: [bold]{LOCATION_CONTEXT}[/bold]")

Running location identification with Pegasus...

1 location marker(s) identified

Location context: St. Louis, MO

## Step 3 — Damage Assessment (Pegasus)
Pegasus analyzes every frame of the video and produces a structured damage inventory.

In [ ]:
DISASTER_CONTEXT = {
    "tornado":    "tornado or severe windstorm",
    "hurricane":  "hurricane or tropical storm",
    "wildfire":   "wildfire or forest fire",
    "flood":      "flooding or flash flood",
    "earthquake": "earthquake",
    "auto":       "natural disaster",
}
disaster_label = DISASTER_CONTEXT.get(DISASTER_TYPE, "natural disaster")

DAMAGE_PROMPT = f"""
You are an expert disaster damage assessor reviewing aerial drone footage captured after a {disaster_label}
over {LOCATION_CONTEXT}.

Carefully analyze every part of this footage and produce a structured damage inventory.
For each distinct damaged area or structure you observe, provide:

1. DAMAGE_TYPE: (choose one) missing_roof | partial_roof_damage | structural_collapse |
   fire_damage | flood_submersion | flood_debris | road_blocked | bridge_damage |
   downed_trees | downed_power_lines | vehicle_damage | other
2. SEVERITY: minor | moderate | severe | complete_destruction
3. STRUCTURE_TYPE: residential_home | commercial_building | road | bridge |
   vehicle | utility_infrastructure | other
4. LOCATION: The most specific real-world location — use street names, intersections,
   block numbers, or landmark references visible in the footage.
5. DESCRIPTION: One to two sentence description of the damage.
6. TIMESTAMP_APPROX: Approximate time in the video (e.g., "0:10-0:30").

Format as a JSON array of objects with keys:
damage_type, severity, structure_type, location, description, timestamp_approx

Only output the JSON array. No preamble or commentary.
"""

console.print("[blue]Running damage assessment with Pegasus...[/blue]")
damage_response = client.analyze(video_id=VIDEO_ID, prompt=DAMAGE_PROMPT)
damage_text = damage_response.data

json_match = re.search(r'\[.*\]', damage_text, re.DOTALL)
if json_match:
    damage_items = json.loads(json_match.group())
else:
    console.print("[yellow]Could not parse JSON; storing raw response[/yellow]")
    damage_items = [{"raw_response": damage_text}]

console.print(f"[green]Identified {len(damage_items)} damage event(s)[/green]")
print(json.dumps(damage_items, indent=2))

Running damage assessment with Pegasus...

In [ ]:
RESOURCE_PROMPT = f"""
You are an emergency logistics coordinator reviewing aerial drone footage of a {disaster_label}
impact zone in {LOCATION_CONTEXT}.

Based on all visible damage in this footage, produce a prioritized resource manifest.
For each resource need you identify, provide:

1. RESOURCE: The specific supply or service needed (e.g., "heavy-duty tarps", "water pumps",
   "sandbags", "chainsaw crews", "structural engineers", "hazmat team", "generators")
2. QUANTITY_ESTIMATE: Best estimate of quantity needed (e.g., "20-30 units", "2 crews", "500 bags")
3. TRIGGERED_BY: The damage condition that requires this resource
4. LOCATION: The most specific location for delivery — street name, intersection, or landmark
5. PRIORITY: immediate (0-24h) | short_term (1-7 days) | long_term (1-4 weeks)
6. NOTES: Any special considerations (e.g., "requires heavy equipment access", "standing water present")

Format as a JSON array of objects with keys:
resource, quantity_estimate, triggered_by, location, priority, notes

Only output the JSON array. No preamble or commentary.
"""

console.print("[blue]Running resource analysis with Pegasus...[/blue]")
resource_response = client.analyze(video_id=VIDEO_ID, prompt=RESOURCE_PROMPT)
resource_text = resource_response.data

json_match = re.search(r'\[.*\]', resource_text, re.DOTALL)
if json_match:
    resource_items = json.loads(json_match.group())
else:
    console.print("[yellow]Could not parse JSON; storing raw response[/yellow]")
    resource_items = [{"raw_response": resource_text}]

console.print(f"[green]Identified {len(resource_items)} resource need(s)[/green]")
print(json.dumps(resource_items, indent=2))

## Step 4 — Resource Requirements (Pegasus)
Maps damage findings to specific emergency resources and supplies.

In [ ]:
ACCESS_PROMPT = f"""
You are a search-and-rescue operations planner reviewing aerial drone footage of a {DISASTER_TYPE}
impact zone in {LOCATION_CONTEXT}.

Analyze all roads, paths, bridges, and access corridors visible in this footage.
For each access finding, provide:

1. LOCATION_DESC: The most specific real-world location — name the street, intersection, or
   landmark. Use St. Louis street names where visible (e.g., "Kingshighway at Forest Park Ave",
   "I-64 eastbound on-ramp near Grand Blvd"). If street names are not readable, describe
   relative to a visible landmark.
2. STATUS: passable | partially_blocked | fully_blocked | unknown
3. OBSTRUCTION_TYPE: (if blocked) flooding | debris | structural_collapse | downed_trees |
   downed_power_lines | fire | mud_landslide | none
4. OBSTRUCTION_DETAIL: Describe what is blocking access and approximate extent.
5. ALTERNATE_ROUTE: If blocked, name a specific alternate street or route visible in the footage.
   If no alternate is visible, say "not visible in footage".

Also identify any viable helicopter or drone landing zones, and any open areas suitable for
emergency staging. Name the location as specifically as possible.

Format your entire response as a JSON object with two keys:
- "access_points": array of objects with keys: location_desc, status, obstruction_type,
  obstruction_detail, alternate_route
- "staging_areas": array of strings describing viable staging/landing zones with specific locations

Only output the JSON object. No preamble or commentary.
"""

console.print("[blue]Running access/route analysis with Pegasus...[/blue]")
access_response = client.analyze(video_id=VIDEO_ID, prompt=ACCESS_PROMPT)
access_text = access_response.data

json_match = re.search(r'\{.*\}', access_text, re.DOTALL)
if json_match:
    access_data = json.loads(json_match.group())
else:
    console.print("[yellow]Could not parse JSON; storing raw response[/yellow]")
    access_data = {"access_points": [], "staging_areas": [], "raw_response": access_text}

console.print(f"[green]Access analysis complete[/green]")
print(json.dumps(access_data, indent=2))

## Step 5 — Access & Route Analysis (Pegasus + Marengo Search)
Identifies road blockages, access obstructions, and viable alternate routes. Uses Marengo to surface specific scenes.

In [ ]:
# --- Marengo Semantic Search: surface specific obstruction scenes ---
OBSTRUCTION_QUERIES = [
    "blocked road debris",
    "flooded road underwater",
    "collapsed bridge",
    "downed power lines road",
    "fire blocking access",
    "open field landing zone",
]

console.print("[blue]Running Marengo scene search for obstructions...[/blue]")
marengo_results = {}

for query in OBSTRUCTION_QUERIES:
    results = client.search.query(
        index_id=index.id,
        query_text=query,
        search_options=["visual", "audio"],
    )
    # Collect top 3 results per query (results are already ranked by relevance)
    clips = []
    for item in results:
        clips.append({
            "start": item.start,
            "end": item.end,
            "rank": item.rank,
        })
        if len(clips) >= 3:
            break
    if clips:
        marengo_results[query] = clips
        console.print(f"  [cyan]{query}[/cyan]: {len(clips)} clip(s) found")
    else:
        console.print(f"  [dim]{query}: no matches[/dim]")

print(json.dumps(marengo_results, indent=2))

## Step 6 — Financial Damage Estimation
Pegasus counts damage instances; we apply unit cost ranges from FEMA/insurance data to produce an estimate.

In [ ]:
FINANCIAL_PROMPT = """
You are a disaster damage appraiser reviewing aerial drone footage.

Count every instance of each damage type you can observe. Be as specific as possible.
Provide your response as a JSON object with exactly these keys and integer values
(use 0 if none observed, use your best estimate for ranges):

{
  "complete_roof_loss": <count of buildings with total roof missing>,
  "partial_roof_damage": <count of buildings with partial roof damage>,
  "structural_collapse": <count of fully collapsed structures>,
  "fire_damage_total_loss": <count of structures burned to foundation>,
  "fire_damage_partial": <count of structures with significant fire/smoke damage>,
  "flooded_structures": <count of structures with visible flood water inside or surrounding>,
  "flooded_road_segments": <count of distinct road segments underwater or impassable>,
  "damaged_bridges": <count of damaged or collapsed bridges>,
  "damaged_vehicles": <count of vehicles that appear destroyed or heavily damaged>,
  "downed_utility_poles": <count of downed power poles or utility infrastructure>,
  "debris_cleared_acres_needed": <estimated acres of land requiring debris clearing>,
  "assessment_confidence": "low" | "medium" | "high"
}

Only output the JSON object. No commentary.
"""

console.print("[blue]Running damage quantification with Pegasus...[/blue]")
financial_response = client.analyze(video_id=VIDEO_ID, prompt=FINANCIAL_PROMPT)
financial_text = financial_response.data

json_match = re.search(r'\{.*\}', financial_text, re.DOTALL)
damage_counts = json.loads(json_match.group()) if json_match else {}

# Unit cost ranges (low, mid, high) in USD — FEMA and insurance industry averages
UNIT_COSTS = {
    "complete_roof_loss":          (10_000,  18_000,  30_000),
    "partial_roof_damage":         ( 2_500,   6_000,  12_000),
    "structural_collapse":         (80_000, 150_000, 300_000),
    "fire_damage_total_loss":      (90_000, 175_000, 350_000),
    "fire_damage_partial":         (15_000,  50_000, 120_000),
    "flooded_structures":          (20_000,  50_000, 120_000),
    "flooded_road_segments":       ( 5_000,  25_000,  75_000),
    "damaged_bridges":            (100_000, 500_000, 2_000_000),
    "damaged_vehicles":            (15_000,  30_000,  60_000),
    "downed_utility_poles":         (5_000,  12_000,  25_000),
    "debris_cleared_acres_needed":  (3_000,   8_000,  20_000),
}

cost_rows = []
total_low = total_mid = total_high = 0.0

for key, (low, mid, high) in UNIT_COSTS.items():
    count = damage_counts.get(key, 0)
    if not isinstance(count, (int, float)) or count == 0:
        continue
    cost_rows.append({
        "category": key.replace("_", " ").title(),
        "count": count,
        "low":  count * low,
        "mid":  count * mid,
        "high": count * high,
    })
    total_low  += count * low
    total_mid  += count * mid
    total_high += count * high

table = Table(title="Financial Damage Estimate", show_lines=True)
table.add_column("Damage Category", style="bold")
table.add_column("Count", justify="right")
table.add_column("Low",  justify="right", style="green")
table.add_column("Mid",  justify="right", style="yellow")
table.add_column("High", justify="right", style="red")
for row in cost_rows:
    table.add_row(row["category"], str(row["count"]),
                  f"${row['low']:,.0f}", f"${row['mid']:,.0f}", f"${row['high']:,.0f}")
table.add_section()
table.add_row("[bold]TOTAL[/bold]", "",
              f"[bold green]${total_low:,.0f}[/bold green]",
              f"[bold yellow]${total_mid:,.0f}[/bold yellow]",
              f"[bold red]${total_high:,.0f}[/bold red]")
console.print(table)

In [ ]:
def format_report(
    location: str,
    disaster: str,
    geo_items: list,
    damage_items: list,
    resource_items: list,
    access_data: dict,
    marengo_results: dict,
    cost_rows: list,
    total_low: float,
    total_mid: float,
    total_high: float,
    confidence: str,
) -> str:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")
    lines = [
        f"# DISASTER ASSESSMENT REPORT",
        f"**Location:** {location}",
        f"**Disaster Type:** {disaster.title()}",
        f"**Generated:** {ts}",
        f"**Analysis Confidence:** {confidence.upper()}",
        "",
        "---",
        "",
        "## 0. IDENTIFIED LOCATIONS",
        "",
        "| Type | Identifier | Confidence | Timestamp | Notes |",
        "|---|---|---|---|---|",
    ]
    for g in geo_items:
        if "raw_response" in g:
            lines.append(g["raw_response"])
            continue
        lines.append(
            f"| {g.get('type','').replace('_',' ').title()} "
            f"| {g.get('identifier','N/A')} "
            f"| {g.get('confidence','?')} "
            f"| {g.get('timestamp_approx','N/A')} "
            f"| {g.get('notes','')} |"
        )

    lines += ["", "---", "", "## 1. DAMAGE ASSESSMENT"]

    severity_order = {"complete_destruction": 0, "severe": 1, "moderate": 2, "minor": 3}
    sorted_damage = sorted(
        damage_items,
        key=lambda x: severity_order.get(x.get("severity", "minor"), 4)
    )

    for item in sorted_damage:
        if "raw_response" in item:
            lines.append(item["raw_response"])
            continue
        sev   = item.get("severity", "unknown").replace("_", " ").upper()
        dtype = item.get("damage_type", "unknown").replace("_", " ").title()
        stype = item.get("structure_type", "unknown").replace("_", " ").title()
        loc   = item.get("location", "Location not identified")
        ts_approx = item.get("timestamp_approx", "N/A")
        desc  = item.get("description", "")
        lines.append(f"### [{sev}] {dtype} — {stype}")
        lines.append(f"- **Location:** {loc}")
        lines.append(f"- **Timestamp:** {ts_approx}")
        lines.append(f"- {desc}")
        lines.append("")

    lines += [
        "---", "",
        "## 2. RESOURCE REQUIREMENTS", "",
        "| Resource | Quantity | Priority | Location | Triggered By | Notes |",
        "|---|---|---|---|---|---|",
    ]

    priority_order = {"immediate": 0, "short_term": 1, "long_term": 2}
    for r in sorted(resource_items, key=lambda x: priority_order.get(x.get("priority", "long_term"), 3)):
        if "raw_response" in r:
            lines.append(r["raw_response"])
            continue
        lines.append(
            f"| {r.get('resource','N/A')} "
            f"| {r.get('quantity_estimate','N/A')} "
            f"| **{r.get('priority','N/A').replace('_',' ').title()}** "
            f"| {r.get('location','N/A')} "
            f"| {r.get('triggered_by','N/A')} "
            f"| {r.get('notes','')} |"
        )

    lines += ["", "---", "", "## 3. ACCESS & ROUTE ANALYSIS", "", "### Road & Access Conditions"]

    for ap in access_data.get("access_points", []):
        status     = ap.get("status", "unknown").replace("_", " ").upper()
        loc        = ap.get("location_desc", "Unknown location")
        obs_type   = ap.get("obstruction_type", "none").replace("_", " ").title()
        obs_detail = ap.get("obstruction_detail", "None")
        alt        = ap.get("alternate_route", "Not identified")
        lines.append(f"**[{status}]** {loc}")
        if status != "PASSABLE":
            lines.append(f"- Obstruction: {obs_type} — {obs_detail}")
            lines.append(f"- Alternate Route: {alt}")
        lines.append("")

    staging = access_data.get("staging_areas", [])
    if staging:
        lines.append("### Viable Staging / Landing Zones")
        for s in staging:
            lines.append(f"- {s}")
        lines.append("")

    if marengo_results:
        lines.append("### Marengo Video Scene References")
        for query, clips in marengo_results.items():
            lines.append(f"**{query.title()}**")
            for c in clips:
                lines.append(f"  - {c['start']:.1f}s – {c['end']:.1f}s  (rank: {c['rank']})")
        lines.append("")

    lines += [
        "---", "",
        "## 4. FINANCIAL DAMAGE ESTIMATE", "",
        "| Damage Category | Count | Low | Mid | High |",
        "|---|---|---|---|---|",
    ]
    for row in cost_rows:
        lines.append(
            f"| {row['category']} | {row['count']} "
            f"| ${row['low']:,.0f} | ${row['mid']:,.0f} | ${row['high']:,.0f} |"
        )
    lines += [
        f"| **TOTAL** | | **${total_low:,.0f}** | **${total_mid:,.0f}** | **${total_high:,.0f}** |",
        "",
        "> Estimates based on FEMA and insurance industry unit cost averages.",
        f"> Assessment confidence: **{confidence.upper()}**.",
        "> This estimate covers visible damage only — subsurface and hidden damage not included.",
        "",
        "---",
        "*Report generated by Disaster Zone Aerial Assessment System using TwelveLabs Marengo + Pegasus.*",
    ]

    return "\n".join(lines)


report_md = format_report(
    location=LOCATION_LABEL,
    disaster=DISASTER_TYPE,
    geo_items=geo_items,
    damage_items=damage_items,
    resource_items=resource_items,
    access_data=access_data,
    marengo_results=marengo_results,
    cost_rows=cost_rows,
    total_low=total_low,
    total_mid=total_mid,
    total_high=total_high,
    confidence=damage_counts.get("assessment_confidence", "medium"),
)

console.print(Markdown(report_md))

In [ ]:
# Save report to disk
report_filename = f"disaster_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
Path(report_filename).write_text(report_md, encoding="utf-8")

# Save raw JSON data alongside it
json_filename = report_filename.replace(".md", "_data.json")
report_data = {
    "metadata": {
        "location": LOCATION_LABEL,
        "city": LOCATION_CITY,
        "disaster_type": DISASTER_TYPE,
        "video_id": VIDEO_ID,
        "index_id": index.id,
        "generated_at": datetime.now().isoformat(),
    },
    "identified_locations": geo_items,
    "damage_assessment": damage_items,
    "resource_requirements": resource_items,
    "access_analysis": access_data,
    "marengo_scene_matches": marengo_results,
    "damage_counts": damage_counts,
    "financial_estimate": {
        "line_items": cost_rows,
        "total_low_usd":  total_low,
        "total_mid_usd":  total_mid,
        "total_high_usd": total_high,
    },
}
Path(json_filename).write_text(json.dumps(report_data, indent=2), encoding="utf-8")

console.print(f"[green]Report saved:[/green] {report_filename}")
console.print(f"[green]Data saved:[/green]   {json_filename}")
console.print(Panel.fit(
    f"[bold]Assessment Complete[/bold]\n"
    f"Location markers: {len(geo_items)}\n"
    f"Damage events: {len(damage_items)}\n"
    f"Resource needs: {len(resource_items)}\n"
    f"Estimated damage: ${total_low:,.0f} – ${total_high:,.0f}",
    border_style="green"
))